#### DISTRIBUTION PROFILE

##### 04.1 DOCUMENTAÇÃO DO DISTRIBUTION PROFILE

###### Objetivo

O **Distribution Profile** é responsável por analisar como os valores estão distribuídos nas colunas de um DataFrame PySpark, identificando frequências, concentrações, diversidade e dominância.

Enquanto o **Schema Profile** analisa as características estruturais das colunas, o Distribution Profile investiga o comportamento dos valores observados.

O componente é genérico, independente da origem dos dados e não utiliza regras específicas de negócio.

---

###### Fonte de dados

O componente utiliza o DataFrame `df_prepared`, disponibilizado pelo Notebook 02 — Data Preparation.

A análise considera as colunas identificadas como elegíveis em `colunas_distribution`.

O Distribution Profile não altera os dados originais e não realiza limpeza ou correção automática de valores.

---

###### Principais etapas

1. **Validação:** verifica o DataFrame e os parâmetros necessários.
2. **Elegibilidade:** identifica colunas compatíveis com a análise de distribuição.
3. **Resumo estatístico:** calcula métricas descritivas das colunas numéricas.
4. **Frequências:** contabiliza a ocorrência dos valores.
5. **Top N:** identifica os valores mais frequentes por coluna.
6. **Concentração:** calcula a participação acumulada dos principais valores.
7. **Diversidade:** mede a heterogeneidade da distribuição.
8. **Dominância:** identifica a participação do valor mais frequente.
9. **Regularidade:** classifica o nível de concentração observado.
10. **Consolidação:** reúne os indicadores por coluna.
11. **Visualização:** disponibiliza resultados para análise exploratória.
12. **Testes:** verifica a consistência dos resultados, quando habilitados.

---

###### Contrato de entrada

| Entrada | Descrição |
|---|---|
| `df_prepared` | DataFrame PySpark preparado para análise. |
| `total_registros` | Quantidade total de registros. |
| `colunas_distribution` | Colunas elegíveis para análise de distribuição. |
| `TOP_N` | Quantidade de valores destacados no ranking. |
| `TOP_CONCENTRATION_N` | Níveis de concentração analisados. |

---

###### Parâmetros de execução

| Parâmetro | Finalidade |
|---|---|
| `TOP_N` | Define a quantidade de valores no ranking. |
| `TOP_CONCENTRATION_N` | Define os níveis de concentração, como Top 1, Top 3 e Top 5. |
| `DISTRIBUTION_MODE` | Controla a estratégia de cálculo das frequências. |
| `DISTRIBUTION_DISPLAY_RESULTS` | Controla a exibição dos resultados. |
| `DISTRIBUTION_RUN_TESTS` | Habilita testes de validação. |

**Modos de execução:**

- `exact`: calcula frequências exatas para todas as colunas elegíveis.
- `skip_high_cardinality`: identifica colunas de alta cardinalidade por estimativa e permite omitir cálculos de frequência excessivamente custosos.

Quando uma análise é omitida, o resultado deve registrar explicitamente essa condição. Uma métrica não calculada não deve ser interpretada como zero.

---

###### Indicadores calculados

| Indicador | Descrição |
|---|---|
| `preenchidos` | Quantidade de valores não NULL. |
| `media` | Média dos valores numéricos. |
| `desvio_padrao` | Dispersão dos valores numéricos. |
| `minimo` | Menor valor observado. |
| `maximo` | Maior valor observado. |
| `frequencia` | Quantidade de ocorrências de um valor. |
| `pct_valor` | Participação percentual de um valor. |
| `concentracao_top1` | Participação do valor mais frequente. |
| `concentracao_top3` | Participação dos três valores mais frequentes. |
| `concentracao_top5` | Participação dos cinco valores mais frequentes. |
| `diversidade_shannon` | Entropia de Shannon da distribuição. |
| `entropia_normalizada` | Diversidade relativa ao máximo teórico da distribuição observada. |
| `indice_dominancia` | Participação percentual do valor predominante. |
| `classificacao_regularidade` | Categoria descritiva do nível de concentração. |

---

###### Definições dos indicadores

**Frequência percentual**

`pct_valor = frequencia / total_registros × 100`

**Concentração Top N**

`concentracao_topN = soma das frequências dos N valores mais frequentes / total_registros × 100`

**Entropia de Shannon**

`H = - Σ p(x) × log2(p(x))`

Em que `p(x)` representa a proporção de ocorrência de cada valor.

Quanto maior a entropia, maior tende a ser a diversidade observada.

**Entropia normalizada**

`H_normalizada = H / log2(k)`

Em que `k` é a quantidade de valores distintos considerados na distribuição.

Para distribuições com apenas um valor distinto, a entropia normalizada é definida como zero.

**Índice de dominância**

`indice_dominancia = concentracao_top1`

---

###### Classificação da regularidade

| Condição | Classificação |
|---|---|
| Dominância ≥ 80% | ALTAMENTE CONCENTRADA |
| Dominância ≥ 50% e menor que 80% | CONCENTRADA |
| Dominância ≥ 20% e menor que 50% | MODERADAMENTE DISTRIBUÍDA |
| Dominância menor que 20% | DISTRIBUÍDA |

Essas classificações descrevem o comportamento estatístico da distribuição.

Uma coluna altamente concentrada não é necessariamente inadequada. Da mesma forma, uma distribuição diversificada não representa automaticamente maior qualidade.

---

###### Tratamento de valores ausentes

Valores NULL, strings vazias e NaN exigem interpretação específica.

A política de inclusão ou exclusão desses valores deve ser consistente entre os cálculos de frequência, concentração, dominância e diversidade.

O Profile deve permitir distinguir ausência de informação de diversidade real dos valores observados.

A presença de valores ausentes não deve ser confundida automaticamente com um problema de distribuição.

---

###### Escalabilidade e desempenho

O cálculo exato das frequências pode ser custoso em colunas de alta cardinalidade.

Para grandes volumes, o Profile disponibiliza uma estratégia de execução que permite identificar colunas potencialmente caras e omitir seus cálculos de frequência.

Essa estratégia não substitui os resultados exatos por valores estimados sem identificação.

Os indicadores omitidos permanecem explicitamente indisponíveis, preservando a rastreabilidade e evitando conclusões incorretas.

---

###### Contrato de saída

**DataFrame principal: `distribution_profile_df`**

Consolida os indicadores de concentração, diversidade, dominância e regularidade por coluna.

**DataFrames auxiliares:**

| DataFrame | Finalidade |
|---|---|
| `resumo_distribuicao_df` | Resumo estatístico das colunas numéricas. |
| `frequencia_valores_df` | Frequências dos valores analisados. |
| `frequencia_top5` | Ranking dos principais valores. |
| `concentracao_df` | Concentração Top N por coluna. |
| `diversidade_df` | Indicadores de diversidade. |
| `dominancia_df` | Indicador de dominância. |
| `regularidade_df` | Classificação da distribuição. |
| `distribution_summary_df` | Resumo executivo por coluna. |

**DataFrames de visualização:**

- `visualizacao_top_valores_df`
- `visualizacao_concentracao_df`
- `visualizacao_diversidade_dominancia_df`

---

###### Limitações e interpretação

- Frequências exatas podem demandar alto consumo de recursos.
- Colunas com muitos valores distintos podem apresentar elevado custo computacional.
- A entropia depende da diversidade e das proporções dos valores observados.
- As faixas de regularidade são descritivas, não critérios universais de qualidade.
- Distribuições altamente concentradas podem ser naturais em determinados conjuntos de dados.
- Colunas omitidas por custo computacional não devem receber métricas fictícias.
- A análise de distribuição não verifica a validade semântica dos valores.

---

###### Resultado esperado

Produzir um diagnóstico quantitativo e reutilizável da distribuição dos valores, permitindo identificar concentração, diversidade, dominância e comportamentos que merecem investigação.

Os resultados podem apoiar análises exploratórias e compor indicadores posteriores do Data Quality Framework.

> **Princípio arquitetural:** descrever o comportamento estatístico dos dados sem presumir que concentração, diversidade ou predominância representem, isoladamente, boa ou má qualidade.


In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 04.3 PARÂMETROS DO PROFILE
# ============================================================

# O QUE FAZ:
# Define os parâmetros específicos utilizados pelas análises do Distribution Profile.

# COMO FAZ:
# Define a quantidade de valores exibidos nas análises Top N e os níveis utilizados para medir a concentração dos valores.

# POR QUE É IMPORTANTE:
# Mantém as configurações específicas do Distribution Profile próximas às análises que utilizam esses parâmetros, sem sobrecarregar o notebook central de bibliotecas.

# PERGUNTA RESPONDIDA:
# "Quais parâmetros serão utilizados nas análises de distribuição?"

TOP_N = int(globals().get("TOP_N", 5))
TOP_CONCENTRATION_N = list(globals().get("TOP_CONCENTRATION_N", [1, 3, 5]))
DISTRIBUTION_DISPLAY_RESULTS = bool(globals().get("DISPLAY_RESULTS", True))
DISTRIBUTION_RUN_TESTS = False

# exact: todas as frequências; skip_high_cardinality: omite as colunas
# cuja cardinalidade estimada ultrapasse o limite (não aproxima frequências).
DISTRIBUTION_MODE = "exact"
DISTRIBUTION_MAX_DISTINCT = 100000
DISTRIBUTION_CARDINALITY_RSD = 0.05

if TOP_N < 1 or not TOP_CONCENTRATION_N or any(n < 1 for n in TOP_CONCENTRATION_N):
    raise ValueError("TOP_N e TOP_CONCENTRATION_N devem ser positivos.")
if DISTRIBUTION_MODE not in ("exact", "skip_high_cardinality"):
    raise ValueError("DISTRIBUTION_MODE inválido.")
if DISTRIBUTION_MAX_DISTINCT < 1 or not 0 < DISTRIBUTION_CARDINALITY_RSD < 1:
    raise ValueError("Parâmetros de cardinalidade inválidos.")
print(f"Modo: {DISTRIBUTION_MODE}; TOP_N: {TOP_N}; TOP_CONCENTRATION_N: {TOP_CONCENTRATION_N}")

In [0]:
# ============================================================
# 04.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame preparado e as informações necessárias para o Distribution Profile estão disponíveis.

# COMO FAZ:
# Verifica a existência do DataFrame preparado, do total de registros e da lista de colunas elegíveis para análise de distribuição.

# POR QUE É IMPORTANTE:
# Garante que o Profile seja executado sobre uma estrutura válida e evita processamento desnecessário.

# PERGUNTA RESPONDIDA:
# "O DataFrame possui as informações necessárias para executar o Distribution Profile?"

if "df_prepared" not in globals() or not isinstance(df_prepared, DataFrame):
    raise ValueError("df_prepared não é um DataFrame Spark válido.")
if "total_registros" not in globals() or total_registros <= 0:
    raise ValueError("total_registros deve ser maior que zero.")
if "colunas_distribution" not in globals():
    raise ValueError("colunas_distribution não foi disponibilizada.")
if len(df_prepared.columns) != len(set(df_prepared.columns)):
    raise ValueError("Nomes de colunas duplicados não são suportados.")
colunas_distribution = list(dict.fromkeys(colunas_distribution))
tipos_por_coluna = {f.name: f.dataType for f in df_prepared.schema.fields}
if any(c not in tipos_por_coluna for c in colunas_distribution):
    raise ValueError("colunas_distribution contém colunas inexistentes.")
if not colunas_distribution:
    raise ValueError("Nenhuma coluna elegível para Distribution Profile.")
print(f"Registros: {total_registros}; colunas elegíveis: {len(colunas_distribution)}")

In [0]:
# ============================================================
# 04.5 FUNÇÕES LOCAIS E POLÍTICA DE VALORES
# ============================================================

# O QUE FAZ: 
# Normaliza categorias para análise e define schemas estáveis.

# COMO FAZ: 
# Cria expressões Spark e funções locais sem UDF.

# POR QUE É IMPORTANTE: 
# Separa ausências de valores textuais reais.

# PERGUNTA RESPONDIDA: "Como representar valores de tipos diferentes?"

def dq_distribution_value(name, data_type):
    c = dq_col(name)
    s = c.cast("string")
    if isinstance(data_type, (FloatType, DoubleType)):
        nan = F.isnan(c)
    else:
        nan = F.lit(False)
    if isinstance(data_type, StringType):
        empty = c.isNotNull() & (F.trim(c) == "")
    else:
        empty = F.lit(False)
    # Escapa todos os valores literais que poderiam coincidir com marcadores.
    reserved = s.isin("__DQ_NULL__", "__DQ_EMPTY__", "__DQ_NAN__") | s.startswith("__DQ_VALUE__:")
    return (F.when(c.isNull(), F.lit("__DQ_NULL__"))
             .when(nan, F.lit("__DQ_NAN__"))
             .when(empty, F.lit("__DQ_EMPTY__"))
             .when(reserved, F.concat(F.lit("__DQ_VALUE__:"), s))
             .otherwise(s))

frequency_schema = T.StructType([
    T.StructField("coluna", T.StringType(), False),
    T.StructField("valor", T.StringType(), True),
    T.StructField("frequencia", T.LongType(), True),
    T.StructField("pct_valor", T.DoubleType(), True)
])

summary_schema = T.StructType([
    T.StructField("coluna", T.StringType(), False),
    T.StructField("preenchidos", T.LongType(), True),
    T.StructField("media", T.DoubleType(), True),
    T.StructField("desvio_padrao", T.DoubleType(), True),
    T.StructField("minimo", T.DoubleType(), True),
    T.StructField("maximo", T.DoubleType(), True)
])

def dq_distribution_show(df, name):
    if DISTRIBUTION_DISPLAY_RESULTS:
        print(name)
        display(df)
    else:
        print(f"{name}: visualização desabilitada")

In [0]:
# ============================================================
# 04.6 ELEGIBILIDADE E RESUMO ESTATÍSTICO
# ============================================================

# O QUE FAZ: 
# Identifica colunas a calcular e estatísticas numéricas.

# COMO FAZ: 
# Estima cardinalidade apenas no modo restrito e agrega números.

# POR QUE É IMPORTANTE: 
# Distingue métricas omitidas das efetivamente calculadas.

# PERGUNTA RESPONDIDA: "Quais distribuições são viáveis e qual seu resumo?"

if DISTRIBUTION_MODE == "skip_high_cardinality":
    estimates = df_prepared.agg(*[
        F.approx_count_distinct(dq_distribution_value(c, tipos_por_coluna[c]),
                                DISTRIBUTION_CARDINALITY_RSD).alias(f"d_{i}")
        for i, c in enumerate(colunas_distribution)
    ]).first()
    estimativas = {c: estimates[f"d_{i}"] for i, c in enumerate(colunas_distribution)}
else:
    estimativas = {c: None for c in colunas_distribution}

colunas_calculadas = [c for c in colunas_distribution if (
    DISTRIBUTION_MODE == "exact" or estimativas[c] <= DISTRIBUTION_MAX_DISTINCT
)]
colunas_omitidas = [c for c in colunas_distribution if c not in colunas_calculadas]

status_rows = [(c, "CALCULADO" if c in colunas_calculadas else "NAO_CALCULADO_ALTA_CARDINALIDADE",
                estimativas[c]) for c in colunas_distribution]
status_schema = T.StructType([
    T.StructField("coluna", T.StringType(), False),
    T.StructField("status_calculo", T.StringType(), False),
    T.StructField("cardinalidade_estimada", T.LongType(), True)
])
distribution_status_df = spark.createDataFrame(status_rows, status_schema)

numericas = [c for c in colunas_distribution if isinstance(tipos_por_coluna[c], NumericType)]
metricas_numericas = []
for i, c in enumerate(numericas):
    x = dq_col(c).cast("double")
    valido = x.isNotNull() & (~F.isnan(x)) & (F.abs(x) != F.lit(float("inf")))
    v = F.when(valido, x)
    metricas_numericas.extend([
        F.count(v).alias(f"n_{i}_count"), F.avg(v).alias(f"n_{i}_avg"),
        F.stddev(v).alias(f"n_{i}_std"), F.min(v).alias(f"n_{i}_min"),
        F.max(v).alias(f"n_{i}_max")
    ])

if metricas_numericas:
    stat = df_prepared.agg(*metricas_numericas).first()
    resumo_rows = [(c, stat[f"n_{i}_count"], stat[f"n_{i}_avg"],
                    stat[f"n_{i}_std"], stat[f"n_{i}_min"], stat[f"n_{i}_max"])
                   for i, c in enumerate(numericas)]
else:
    resumo_rows = []
resumo_distribuicao_df = spark.createDataFrame(resumo_rows, summary_schema)
print(f"Distribuições calculadas: {len(colunas_calculadas)}; omitidas: {len(colunas_omitidas)}")
dq_distribution_show(resumo_distribuicao_df, "Resumo estatístico numérico")

In [0]:
# ============================================================
# 04.7 FREQUÊNCIAS DOS VALORES
# ============================================================

# O QUE FAZ: 
# Calcula frequências exatas para colunas selecionadas.

# COMO FAZ: 
# Agrupa cada coluna e une resultados em schema único.

# POR QUE É IMPORTANTE: 
# Sustenta Top N, Shannon e concentração.

# PERGUNTA RESPONDIDA: "Quantas vezes cada categoria aparece?"

frequencias = []
for coluna in colunas_calculadas:
    frequencias.append(
        df_prepared.groupBy(dq_distribution_value(coluna, tipos_por_coluna[coluna]).alias("valor"))
        .agg(F.count(F.lit(1)).cast("long").alias("frequencia"))
        .withColumn("coluna", F.lit(coluna))
        .withColumn("pct_valor", F.col("frequencia") * F.lit(100.0 / total_registros))
        .select("coluna", "valor", "frequencia", "pct_valor")
    )
if frequencias:
    from functools import reduce
    frequencia_valores_df = reduce(lambda a, b: a.unionByName(b), frequencias)
else:
    frequencia_valores_df = spark.createDataFrame([], frequency_schema)
# Evita exibir todos os valores distintos de colunas potencialmente enormes.
print("frequencia_valores_df disponível; consulte somente amostras ou agregados.")

In [0]:
# ============================================================
# 04.8 TOP N E CONCENTRAÇÃO
# ============================================================

# O QUE FAZ: 
# Calcula ranking uma única vez e concentrações acumuladas.

# COMO FAZ: Aplica Window sobre as frequências exatas.

# POR QUE É IMPORTANTE: 
# Evita recalcular ordenações desnecessárias.

# PERGUNTA RESPONDIDA: 
# "Quanto os principais valores representam?"

window_top_valores = Window.partitionBy("coluna").orderBy(F.desc("frequencia"), F.asc("valor"))
concentracao_base = frequencia_valores_df.withColumn("ranking", F.row_number().over(window_top_valores))
frequencia_top5 = (concentracao_base.filter(F.col("ranking") <= TOP_N)
                   .orderBy("coluna", "ranking"))

agregacoes_concentracao = [
    F.sum(F.when(F.col("ranking") <= n, F.col("pct_valor")).otherwise(F.lit(0.0)))
    .alias(f"concentracao_top{n}") for n in TOP_CONCENTRATION_N
]
concentracao_df = concentracao_base.groupBy("coluna").agg(*agregacoes_concentracao)
dq_distribution_show(frequencia_top5, "Top N por coluna")

In [0]:
# ============================================================
# 04.9 ÍNDICE DE DIVERSIDADE
# ============================================================

# O QUE FAZ:
# Calcula o Índice de Diversidade de Shannon para cada coluna.

# COMO FAZ:
# Utiliza a proporção de ocorrência de cada valor e calcula a entropia de Shannon:
#
# H = - Σ p(x) * log2(p(x))
#
# Quanto maior o valor de H, maior tende a ser a diversidade observada na distribuição.

# POR QUE É IMPORTANTE:
# Permite avaliar a diversidade da distribuição de forma quantitativa, complementando a análise de concentração.

# PERGUNTA RESPONDIDA:
# "Quão diversificados são os valores presentes em cada coluna?"

prob = F.col("frequencia") / F.lit(float(total_registros))
diversidade_df = (frequencia_valores_df.groupBy("coluna")
    .agg(F.sum(-prob * F.log2(prob)).alias("diversidade_shannon"),
         F.count(F.lit(1)).cast("long").alias("valores_distintos_distribuicao"))
    .withColumn("entropia_normalizada",
        F.when(F.col("valores_distintos_distribuicao") > 1,
               F.col("diversidade_shannon") / F.log2(F.col("valores_distintos_distribuicao")))
         .otherwise(F.lit(0.0))))
dq_distribution_show(diversidade_df, "Diversidade de Shannon")

In [0]:
# ============================================================
# 04.10 DOMINÂNCIA E REGULARIDADE
# ============================================================

# O QUE FAZ:
# Calcula o percentual de participação do valor mais frequente de cada coluna.

# COMO FAZ:
# Utiliza a concentração Top 1 já calculada anteriormente.

# POR QUE É IMPORTANTE:
# A dominância mostra de forma direta o quanto o valor mais frequente representa da distribuição da coluna.

# PERGUNTA RESPONDIDA:
# "Qual é a participação do valor mais frequente em cada coluna?"

if 1 not in TOP_CONCENTRATION_N:
    raise ValueError("TOP_CONCENTRATION_N precisa incluir 1 para dominância.")
dominancia_df = concentracao_df.select("coluna", F.col("concentracao_top1").alias("indice_dominancia"))
regularidade_df = (dominancia_df.withColumn("classificacao_regularidade",
    F.when(F.col("indice_dominancia") >= 80, "ALTAMENTE CONCENTRADA")
     .when(F.col("indice_dominancia") >= 50, "CONCENTRADA")
     .when(F.col("indice_dominancia") >= 20, "MODERADAMENTE DISTRIBUÍDA")
     .otherwise("DISTRIBUÍDA")))
dq_distribution_show(regularidade_df, "Dominância e regularidade")

In [0]:
# ============================================================
# 04.11 CONSOLIDAÇÃO DO DISTRIBUTION PROFILE
# ============================================================

# O QUE FAZ: 
# Consolida métricas e status por coluna elegível.

# COMO FAZ: 
# Parte do inventário e aplica left joins nos resultados.

# POR QUE É IMPORTANTE: 
# Mantém colunas omitidas visíveis, com NULL nas métricas.

# PERGUNTA RESPONDIDA: "Qual é o perfil consolidado de cada coluna?"

distribution_profile_df = (distribution_status_df
    .join(concentracao_df, "coluna", "left")
    .join(diversidade_df, "coluna", "left")
    .join(regularidade_df, "coluna", "left")
    .orderBy("coluna"))
dq_distribution_show(distribution_profile_df, "Distribution Profile consolidado")

In [0]:
# ============================================================
# 04.12 RESUMO EXECUTIVO
# ============================================================

# O QUE FAZ: 
# Oferece uma interpretação descritiva por coluna.

# COMO FAZ: 
# Usa métricas observadas e status explícito.

# POR QUE É IMPORTANTE: 
# Não confunde dados ausentes com baixa diversidade.

# PERGUNTA RESPONDIDA: "Qual comportamento predomina?"

for n in (1, 3, 5):
    if n not in TOP_CONCENTRATION_N:
        raise ValueError("O contrato do resumo exige TOP_CONCENTRATION_N=[1,3,5] no mínimo.")
distribution_summary_df = (distribution_profile_df.select(
    "coluna", "status_calculo", "concentracao_top1", "concentracao_top3",
    "concentracao_top5", "diversidade_shannon", "indice_dominancia",
    "classificacao_regularidade", "entropia_normalizada")
    .withColumn("leitura",
        F.when(F.col("status_calculo") != "CALCULADO", "Distribuição não calculada.")
         .when(F.col("indice_dominancia") >= 80, "Distribuição fortemente concentrada.")
         .when(F.col("indice_dominancia") >= 50, "Distribuição concentrada nos principais valores.")
         .when(F.col("diversidade_shannon") >= 5, "Distribuição com elevada entropia observada.")
         .otherwise("Distribuição relativamente diversificada."))
    .orderBy(F.desc_nulls_last("indice_dominancia")))
dq_distribution_show(distribution_summary_df, "Resumo executivo")

In [0]:
# ============================================================
# 04.13 VISUALIZAÇÕES
# ============================================================

# O QUE FAZ: 
# Prepara visões de Top N, concentração e diversidade.

# COMO FAZ: 
# Seleciona apenas colunas necessárias para gráficos.

# POR QUE É IMPORTANTE: 
# Disponibiliza saídas reutilizáveis no Databricks.

# PERGUNTA RESPONDIDA: "Como comparar as distribuições visualmente?"

visualizacao_top_valores_df = (frequencia_top5
    .select("coluna", "valor", "frequencia", "pct_valor", "ranking")
    .orderBy("coluna", "ranking"))
visualizacao_concentracao_df = (concentracao_df
    .select("coluna", "concentracao_top1", "concentracao_top3", "concentracao_top5")
    .orderBy(F.desc("concentracao_top1")))
visualizacao_diversidade_dominancia_df = (distribution_profile_df
    .select("coluna", "diversidade_shannon", "indice_dominancia", "classificacao_regularidade")
    .orderBy(F.desc_nulls_last("diversidade_shannon")))
for titulo, df_viz in [
    ("Top valores", visualizacao_top_valores_df),
    ("Concentração", visualizacao_concentracao_df),
    ("Diversidade x dominância", visualizacao_diversidade_dominancia_df)
]:
    dq_distribution_show(df_viz, titulo)

In [0]:
# ============================================================
# 04.14 TESTES OPCIONAIS
# ============================================================

# O QUE FAZ: 
# Valida invariantes das métricas de distribuição.

# COMO FAZ: 
# Executa asserts somente quando habilitado.

# POR QUE É IMPORTANTE: 
# Identifica inconsistências sem alterar a fonte.

# PERGUNTA RESPONDIDA: "As métricas respeitam invariantes básicas?"

if DISTRIBUTION_RUN_TESTS:
    for linha in frequencia_valores_df.groupBy("coluna").agg(
        F.sum("frequencia").alias("total")
    ).collect():
        assert linha["total"] == total_registros, f"Frequência inconsistente: {linha['coluna']}"
    for linha in concentracao_df.collect():
        vals = [linha[f"concentracao_top{n}"] for n in sorted(TOP_CONCENTRATION_N)]
        assert all(-1e-8 <= v <= 100.000001 for v in vals)
        assert vals == sorted(vals), f"Concentração não monotônica: {linha['coluna']}"
    print("Testes de invariantes concluídos.")
else:
    print("Testes opcionais desabilitados.")